In [ ]:
# CELL 1 — KAGGLE MODEL FACTORY CONFIGURATION
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

# Defaults are used by Kaggle's fresh Save & Run All session.
DATASET_INDEX = 0
MODEL_DEFAULT = "Compare top models"
OPTIMIZATION_DEFAULT = "fast"   # fast, balanced, deep
TARGET_DEFAULT = "AUTO"
TASK = "auto"  # auto / classification / regression
MAX_ROWS = 30000
TEST_SIZE = 0.20
CV_FOLDS = 3
SEED = 42
SAVE_TOP_N = 3
OUTPUT_DIR = "/kaggle/working/model_factory"

MODEL_INFO = {
    "Random Forest": ("random_forest", "Tree ensemble for nonlinear classification or regression. Useful for fraud, equipment failure, churn, and pricing."),
    "Extra Trees": ("extra_trees", "Highly randomized tree ensemble for tabular classification or regression. Useful as a fast competitor to Random Forest."),
    "Hist Gradient Boosting": ("hist_gradient_boosting", "Boosted trees for complex tabular relationships. Useful for operational risk and demand forecasting."),
    "Logistic / Ridge": ("linear", "Regularized linear baseline: Logistic Regression for classes, Ridge for numbers. Useful to establish interpretability and a performance floor."),
    "XGBoost": ("xgboost", "Strong gradient-boosted trees for tabular classification or regression. Useful for fraud detection, delays, and churn."),
    "LightGBM": ("lightgbm", "Efficient histogram boosting for larger tabular datasets. Useful for transactions and demand models."),
    "CatBoost": ("catboost", "Boosted trees well suited to categorical-heavy business records. Useful for banking, customer, and retail datasets."),
    "Compare top models": ("top", "Train Random Forest, Extra Trees, and XGBoost on the same dataset. Compare cross-validation performance and test metrics."),
    "Compare all models": ("all", "Benchmark all seven supported families on the same data. Costs more runtime but broadens architecture comparison."),
}
MODEL_GROUPS = {
    "top": ["random_forest", "extra_trees", "xgboost"],
    "all": ["random_forest", "extra_trees", "hist_gradient_boosting", "linear", "xgboost", "lightgbm", "catboost"],
}
paths = sorted(Path("/kaggle/input").rglob("*.csv")) if Path("/kaggle/input").exists() else []
if not paths:
    print("No attached CSVs. Kaggle: Add Input → attach a Dataset or Notebook Output, then rerun Cell 1.")
else:
    dataset_widget = widgets.Dropdown(options=[(str(p.relative_to('/kaggle/input')), str(p)) for p in paths], index=min(DATASET_INDEX, len(paths)-1), description="Dataset:", layout=widgets.Layout(width="95%"), style={'description_width':'100px'})
    model_widget = widgets.Dropdown(options=list(MODEL_INFO), value=MODEL_DEFAULT, description="Model:", layout=widgets.Layout(width="95%"), style={'description_width':'100px'})
    optimization_widget = widgets.Dropdown(options=['fast','balanced','deep'], value=OPTIMIZATION_DEFAULT, description='Optimize:', style={'description_width':'100px'})
    target_widget = widgets.Text(value=TARGET_DEFAULT, description='Target:', style={'description_width':'100px'})
    help_widget = widgets.HTML()
    def update_help(change=None):
        help_widget.value = '<div style="padding:8px 0"><b>Use case:</b> '+MODEL_INFO[model_widget.value][1]+'</div>'
    model_widget.observe(update_help, names='value')
    update_help()
    display(dataset_widget, model_widget, help_widget, optimization_widget, target_widget)
    print('Choose options and run Cell 2. For Save & Run All, edit the defaults above to match your choices.')


In [ ]:
# CELL 2 — TRAIN, OPTIMIZE, COMPARE, EXPORT
import json, zipfile, time, platform, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier, ExtraTreesRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, mean_absolute_error, mean_squared_error, r2_score

if not paths:
    raise RuntimeError('Attach an input CSV in Kaggle (Add Input), then rerun Cell 1.')
source = dataset_widget.value
selection = MODEL_INFO[model_widget.value][0]
models = MODEL_GROUPS.get(selection, [selection])
search_mode = optimization_widget.value
target_choice = target_widget.value.strip()
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(source)
if df.empty or df.columns.duplicated().any():
    raise ValueError('Dataset is empty or has duplicate columns.')
if MAX_ROWS and len(df)>MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)

known_targets = {
    'flight_delays':'delay_minutes','maintenance':'needs_maintenance','transaction_fraud':'is_fraud','loan_default':'loan_default',
    'appointment_no_show':'no_show','readmission_risk':'readmitted_30d','customer_churn':'churned','demand_forecast':'units_demanded',
    'equipment_failure':'failure_next_7d','quality_defect':'defect_found','late_delivery':'late_delivery','shipment_damage':'damaged',
    'suspicious_login':'suspicious_login','alert_severity':'severity_level','high_latency':'high_latency','capacity_forecast':'requests_next_hour',
    'peak_load':'demand_kw','equipment_outage':'outage_next_30d','property_price':'price_usd','long_listing':'listing_over_90d',
}
regression_targets={'delay_minutes','units_demanded','requests_next_hour','demand_kw','price_usd','latency_ms'}
if target_choice.upper()=='AUTO':
    matches=list(dict.fromkeys([v for k,v in known_targets.items() if k in Path(source).stem.lower() and v in df.columns]))
    if not matches:
        matches=list(dict.fromkeys([v for v in known_targets.values() if v in df.columns]))
    if len(matches)!=1:
        raise ValueError(f'Cannot infer one target ({matches}). Enter the target column in Cell 1.')
    target=matches[0]
else:
    target=target_choice
if target not in df: raise ValueError(f'Missing target: {target}')
df=df.dropna(subset=[target]).copy()
if len(df)<60: raise ValueError('Need at least 60 labeled rows.')
task = ('regression' if target in regression_targets else 'classification') if TASK=='auto' else TASK
if task not in ('regression','classification'): raise ValueError('TASK must be auto/classification/regression.')
exclude={target} | {c for c in df if c.lower() in ('record_id','id','uuid') or c.lower().endswith('_id')}
exclude |= {c for c in df if c != target and c in known_targets.values()}
if target=='high_latency': exclude.add('latency_ms')
X=df.drop(columns=list(exclude),errors='ignore').copy()
# Drop unsupported date/time fields (set calendar-derived predictors explicitly in future versions).
for c in X.select_dtypes(include=['datetime','datetimetz']).columns: X=X.drop(columns=c)
y=df[target]
label_encoder=None
if task=='classification':
    label_encoder=LabelEncoder()
    y=pd.Series(label_encoder.fit_transform(y.astype(str)),index=y.index)
    if y.nunique()<2: raise ValueError('Classification needs 2+ classes.')
else:
    y=pd.to_numeric(y,errors='coerce')
    valid=y.notna(); X=X.loc[valid]; y=y.loc[valid]
    if len(y)<60: raise ValueError('Need at least 60 numeric targets.')

numeric=X.select_dtypes(include=np.number).columns.tolist()
others=[c for c in X if c not in numeric]
text_cols=[c for c in others if any(word in c.lower() for word in ['note','description','feedback','comment','narrative','report','log','text','event'])]
cat_cols=[c for c in others if c not in text_cols]
# Treat text and categoricals as strings before fitting; avoid missing values in TF-IDF.
for c in text_cols: X[c]=X[c].fillna('').astype(str)
for c in cat_cols: X[c]=X[c].fillna('missing').astype(str)
high_card=[c for c in cat_cols if X[c].nunique()>min(300,max(30,len(X)//2))]
X=X.drop(columns=high_card); cat_cols=[c for c in cat_cols if c not in high_card]
transformers=[]
if numeric:
    transformers.append(('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric))
if cat_cols:
    transformers.append(('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols))
# ColumnTransformer passes a Series when the column selector is a string, ideal for TF-IDF.
for i,c in enumerate(text_cols):
    transformers.append((f'text_{i}',TfidfVectorizer(max_features=200,ngram_range=(1,2)),c))
if not transformers: raise ValueError('No usable features.')
preprocessor=ColumnTransformer(transformers,sparse_threshold=0)
stratify=None
if task=='classification':
    class_counts=y.value_counts(); n_classes=len(class_counts)
    if class_counts.min()<2: raise ValueError('Every class needs >=2 rows; generate more data.')
    if min(class_counts.min(),int(np.ceil(len(y)*TEST_SIZE)))>=2 and int(np.ceil(len(y)*TEST_SIZE))>=n_classes:
        stratify=y
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=TEST_SIZE,random_state=SEED,stratify=stratify)
if task=='classification':
    if y_train.value_counts().min()<2: raise ValueError('Insufficient class samples in training split.')
    cv_n=min(CV_FOLDS,int(y_train.value_counts().min()))
    cv=StratifiedKFold(n_splits=cv_n,shuffle=True,random_state=SEED)
    score='f1_weighted'
else:
    cv_n=min(CV_FOLDS,len(y_train)); cv=KFold(n_splits=cv_n,shuffle=True,random_state=SEED)
    score='neg_mean_absolute_error'
if cv_n<2: raise ValueError('Cannot run >=2 cross-validation folds.')

def factory(name):
    reg=task=='regression'
    if name=='random_forest':
        cls=RandomForestRegressor if reg else RandomForestClassifier
        return cls(n_estimators=150,random_state=SEED,n_jobs=-1),{'model__max_depth':[None,8,16],'model__min_samples_leaf':[1,2,4]}
    if name=='extra_trees':
        cls=ExtraTreesRegressor if reg else ExtraTreesClassifier
        return cls(n_estimators=150,random_state=SEED,n_jobs=-1),{'model__max_depth':[None,8,16],'model__min_samples_leaf':[1,2,4]}
    if name=='hist_gradient_boosting':
        cls=HistGradientBoostingRegressor if reg else HistGradientBoostingClassifier
        return cls(random_state=SEED),{'model__max_iter':[100,200],'model__learning_rate':[0.03,0.1]}
    if name=='linear':
        return (Ridge(),{'model__alpha':[0.1,1,10]}) if reg else (LogisticRegression(max_iter=2000),{'model__C':[0.1,1,10]})
    if name=='xgboost':
        from xgboost import XGBClassifier,XGBRegressor
        cls=XGBRegressor if reg else XGBClassifier
        return cls(n_estimators=150,random_state=SEED,n_jobs=2,tree_method='hist'),{'model__max_depth':[3,5,8],'model__learning_rate':[0.03,0.1]}
    if name=='lightgbm':
        from lightgbm import LGBMClassifier,LGBMRegressor
        cls=LGBMRegressor if reg else LGBMClassifier
        return cls(n_estimators=150,random_state=SEED,verbose=-1,n_jobs=2),{'model__num_leaves':[15,31,63],'model__learning_rate':[0.03,0.1]}
    if name=='catboost':
        from catboost import CatBoostClassifier,CatBoostRegressor
        cls=CatBoostRegressor if reg else CatBoostClassifier
        return cls(iterations=150,random_seed=SEED,verbose=False,thread_count=2),{'model__depth':[4,6,8],'model__learning_rate':[0.03,0.1]}
    raise ValueError(name)

iterations={'fast':0,'balanced':6,'deep':20}[search_mode]
trained={}; rows=[]; failures={}
print(f'Dataset: {source}\nTarget: {target} ({task})\nRows: {len(df)}\nFeatures: {len(X.columns)}\nModels: {models}')
for name in models:
    start=time.perf_counter()
    try:
        estimator,params=factory(name)
        pipe=Pipeline([('preprocess',preprocessor),('model',estimator)])
        if iterations:
            candidate_count=int(np.prod([len(v) for v in params.values()]))
            search=RandomizedSearchCV(pipe,params,n_iter=min(iterations,candidate_count),cv=cv,scoring=score,refit=True,random_state=SEED,n_jobs=1,error_score='raise')
            search.fit(X_train,y_train); fitted=search.best_estimator_; cv_score=float(search.best_score_); tuned=search.best_params_
        else:
            cv_score=float(cross_val_score(pipe,X_train,y_train,scoring=score,cv=cv,n_jobs=1,error_score='raise').mean())
            fitted=pipe.fit(X_train,y_train); tuned={}
        pred=fitted.predict(X_test)
        metrics=({'accuracy':float(accuracy_score(y_test,pred)),'f1_weighted':float(f1_score(y_test,pred,average='weighted')),'balanced_accuracy':float(balanced_accuracy_score(y_test,pred))} if task=='classification' else {'MAE':float(mean_absolute_error(y_test,pred)),'RMSE':float(np.sqrt(mean_squared_error(y_test,pred))),'R2':float(r2_score(y_test,pred))})
        rows.append({'model':name,'cv_score':cv_score,'train_seconds':round(time.perf_counter()-start,2),**metrics,'best_params':json.dumps(tuned)})
        trained[name]=fitted
        print(name, 'OK:',metrics)
    except Exception as exc:
        failures[name]=f'{type(exc).__name__}: {exc}'
        print(name,'SKIPPED:',failures[name])
if not rows: raise RuntimeError(f'All model fits failed: {failures}')
leaderboard=pd.DataFrame(rows).sort_values('cv_score',ascending=False).reset_index(drop=True)
display(leaderboard)
leaderboard.to_csv(out/'leaderboard.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(leaderboard['model'],leaderboard['cv_score']); ax.set_title(f'Cross-validation: {score}'); ax.tick_params(axis='x',rotation=35); plt.tight_layout(); fig.savefig(out/'model_comparison.png',dpi=125); plt.show(); plt.close(fig)
# CV chooses models; test scores are reported separately and do not affect model selection.
winners=leaderboard['model'].head(max(1,SAVE_TOP_N)).tolist()
files=[]
for name in winners:
    file=out/f'{name}_pipeline.joblib'; joblib.dump(trained[name],file); files.append(file)
sample=X_test.head(min(5,len(X_test))).copy()
sample.to_csv(out/'sample_inference_input.csv',index=False)
champion=winners[0]
reloaded=joblib.load(out/f'{champion}_pipeline.joblib')
pred=reloaded.predict(sample)
if label_encoder is not None: pred=label_encoder.inverse_transform(pred.astype(int))
(out/'sample_predictions.json').write_text(json.dumps({'model':champion,'predictions':pd.Series(pred).tolist()},indent=2,default=str))
metadata={'source_csv':source,'target':target,'task':task,'model_selection_metric':score,'champion':champion,'winners':winners,'feature_columns':list(X.columns),'numeric_columns':numeric,'categorical_columns':cat_cols,'text_columns':text_cols,'dropped_columns':sorted(exclude|set(high_card)),'classes':label_encoder.classes_.tolist() if label_encoder is not None else None,'seed':SEED,'optimization':search_mode,'generated_utc':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),'sklearn':sklearn.__version__,'failures':failures,'warning':'Only load joblib from trusted sources. For production validate on independent real-world data.'}
(out/'manifest.json').write_text(json.dumps(metadata,indent=2,default=str))
(out/'README.txt').write_text('Kaggle Model Factory artifacts. Load *_pipeline.joblib with joblib.load() in a trusted compatible Python environment; pass a pandas DataFrame with the feature_columns listed in manifest.json. For classification, numeric predictions map to manifest classes in index order. Synthetic scores are not real-world validation.\n')
zip_path=Path('/kaggle/working/model_factory_artifacts.zip'); zip_path.parent.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.iterdir():
        if p.is_file(): z.write(p,arcname=p.name)
print('\nCHAMPION:',champion,'\nFiles:',*[str(p) for p in sorted(out.iterdir())],sep='\n')
print('ZIP:',zip_path)
print('Save a Kaggle notebook version (Save & Run All) to retain files in its Output tab. Publishing in Kaggle Models is a separate step.')
